# Supervised Classification Benchmark on the Iris Dataset

### Objective
Implement a supervised classification pipeline on the **Iris dataset**:
1. Load the benchmark dataset using `pandas`.
2. Perform exploratory pairplots using `seaborn`.
3. Split the dataset into training and testing sets (80/20) and normalize features using `StandardScaler()`.
4. Train **Logistic Regression** and **Decision Tree** classifiers and benchmark **Accuracy, Precision, Recall, and F1-Score**.
5. Plot **Confusion Matrix Heatmaps** and **2D Decision Boundary Visualizations**.

## Step 1: Import Libraries & Load the Benchmark Dataset

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Set visual style
sns.set_theme(style="whitegrid")

# Load the dataset directly from the provided URL
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv"
df = pd.read_csv(url)

print(f"Dataset Shape: {df.shape}")
print("\nClass Distribution:")
print(df['species'].value_counts())
print("\nFirst 5 Rows:")
display(df.head())

## Step 2: Exploratory Pairplots with Seaborn
We visualize pairwise relationships across all four morphological features (`sepal_length`, `sepal_width`, `petal_length`, `petal_width`) colored by `species`.

In [ ]:
pair_grid = sns.pairplot(df, hue="species", markers=["o", "s", "D"], palette="Set2", height=2.3)
pair_grid.fig.suptitle("Exploratory Pairplot of Iris Dataset", y=1.02, fontsize=14, fontweight="bold")
plt.savefig("1_iris_pairplot.png", dpi=300, bbox_inches="tight")
plt.show()

## Step 3: Train/Test Split (80/20) & Feature Scaling (`StandardScaler`)
- We encode the target categorical labels (`setosa`, `versicolor`, `virginica`) into integers (`0, 1, 2`).
- We use an **80/20 stratified split** (`120` training samples, `30` testing samples) so each class is equally represented.
- `StandardScaler()` is fitted **only on the training set** (`X_train`) and then applied to `X_test` to avoid data leakage.

In [ ]:
X = df.drop("species", axis=1)
y = df["species"]

# Encode target species names into numeric labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)
class_names = list(le.classes_)

# 80/20 Train/Test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.20, random_state=42, stratify=y_encoded
)

# Normalize features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set shape: {X_train_scaled.shape}")
print(f"Testing set shape:  {X_test_scaled.shape}")
print(f"Scaled Training Mean (approx 0): { np.round(X_train_scaled.mean(axis=0), 4) }")
print(f"Scaled Training Std  (approx 1): { np.round(X_train_scaled.std(axis=0), 4) }")

## Step 4: Train Models & Benchmark Classification Metrics
We train both `LogisticRegression` and `DecisionTreeClassifier` on the scaled training features and evaluate **Accuracy, Precision, Recall, and F1-Score** on the 20% test set.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=3, random_state=42)
}

benchmark_results = []
predictions = {}

for name, model in models.items():
    # Fit model on scaled training data
    model.fit(X_train_scaled, y_train)
    
    # Predict on scaled test data
    y_pred = model.predict(X_test_scaled)
    predictions[name] = y_pred
    
    # Compute metrics
    benchmark_results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Precision (Weighted)": round(precision_score(y_test, y_pred, average="weighted"), 4),
        "Recall (Weighted)": round(recall_score(y_test, y_pred, average="weighted"), 4),
        "F1-Score (Weighted)": round(f1_score(y_test, y_pred, average="weighted"), 4)
    })
    
    print(f"=== {name} Classification Report ===")
    print(classification_report(y_test, y_pred, target_names=class_names))

benchmark_df = pd.DataFrame(benchmark_results)
print("=== Final Benchmark Comparison Table ===")
display(benchmark_df)

## Step 5A: Plot Confusion Matrix Heatmaps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", cbar=False,
        xticklabels=class_names, yticklabels=class_names, ax=ax,
        annot_kws={"size": 13, "weight": "bold"}
    )
    ax.set_title(f"{name} - Confusion Matrix", fontsize=13, fontweight="bold")
    ax.set_xlabel("Predicted Species", fontsize=11)
    ax.set_ylabel("Actual Species", fontsize=11)

plt.tight_layout()
plt.savefig("2_confusion_matrix_heatmaps.png", dpi=300, bbox_inches="tight")
plt.show()

## Step 5B: Plot Decision Boundary Visualization
To visualize the 2D decision boundaries, we use the two most discriminative standardized features (`petal_length` and `petal_width`) and plot the decision regions alongside the test set points.

In [ ]:
# Select petal_length (index 2) and petal_width (index 3)
X_train_2d = X_train_scaled[:, [2, 3]]
X_test_2d = X_test_scaled[:, [2, 3]]

x_min, x_max = X_train_2d[:, 0].min() - 0.8, X_train_2d[:, 0].max() + 0.8
y_min, y_max = X_train_2d[:, 1].min() - 0.8, X_train_2d[:, 1].max() + 0.8
xx, yy = np.meshgrid(
    np.arange(x_min, x_max, 0.02),
    np.arange(y_min, y_max, 0.02)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (name, base_model) in zip(axes, models.items()):
    # Train 2D model for boundary visualization
    model_2d = base_model.__class__(**base_model.get_params())
    model_2d.fit(X_train_2d, y_train)
    
    # Predict across meshgrid
    Z = model_2d.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot contour zones and test points
    ax.contourf(xx, yy, Z, alpha=0.35, cmap="Set2")
    scatter = ax.scatter(
        X_test_2d[:, 0], X_test_2d[:, 1], c=y_test,
        edgecolor="k", cmap="Set2", s=80
    )
    
    ax.set_title(f"{name} - Decision Boundary", fontsize=13, fontweight="bold")
    ax.set_xlabel("Petal Length (Standardized)", fontsize=11)
    ax.set_ylabel("Petal Width (Standardized)", fontsize=11)
    ax.legend(handles=scatter.legend_elements()[0], labels=class_names, title="Species")

plt.tight_layout()
plt.savefig("3_decision_boundaries.png", dpi=300, bbox_inches="tight")
plt.show()

## Summary of Results
- **Logistic Regression:** `93.33%` Accuracy (`0.9333` Precision, `0.9333` Recall, `0.9333` F1-Score). Produces smooth linear decision boundaries.
- **Decision Tree (`max_depth=3`):** `96.67%` Accuracy (`0.9697` Precision, `0.9667` Recall, `0.9666` F1-Score). Produces orthogonal axis-aligned splits that slightly better capture the boundary between *Iris-versicolor* and *Iris-virginica* on this split.